In [17]:
import warnings
import torch
import torch.autograd.functional as AF
from torch import Tensor
print('PyTorch version:', torch.__version__)

PyTorch version: 2.13.0+cpu


In [18]:
x = torch.arange(1.0, 5.0, requires_grad=True)
y = torch.arange(5.0, 9.0, requires_grad=True)
print(x)
print(y)

tensor([1., 2., 3., 4.], requires_grad=True)
tensor([5., 6., 7., 8.], requires_grad=True)


In [19]:
q=x.dot(y)
z=q.sin()
print(q)
print(z)
print('z.requires_grad:', z.requires_grad)

tensor(70., grad_fn=<DotBackward0>)
tensor(0.7739, grad_fn=<SinBackward0>)
z.requires_grad: True


In [20]:
print('z.grad_fn:', z.grad_fn.name())
print('q.grad_fn:', q.grad_fn.name())
print('x.grad_fn:', x.grad_fn)
print('y.grad_fn:', y.grad_fn)

z.grad_fn: SinBackward0
q.grad_fn: DotBackward0
x.grad_fn: None
y.grad_fn: None


In [21]:
z.backward()

In [22]:
print('x.grad:', x.grad)
print('y.grad:', y.grad)

x.grad: tensor([3.1666, 3.7999, 4.4332, 5.0666])
y.grad: tensor([0.6333, 1.2666, 1.9000, 2.5333])


In [26]:
expected_x_grad = y * x.dot(y).cos()
expected_y_grad = x * x.dot(y).cos()
expected_x_grad = y * x.dot(y).cos()
expected_y_grad = x * x.dot(y).cos()

# 打印自动求导的结果
print("自动求导 x.grad:", x.grad)
print("手算期望 x_grad:", expected_x_grad)


print("自动求导 y.grad:", y.grad)
print("手算期望 y_grad:", expected_y_grad)


自动求导 x.grad: tensor([3.1666, 3.7999, 4.4332, 5.0666])
手算期望 x_grad: tensor([3.1666, 3.7999, 4.4332, 5.0666], grad_fn=<MulBackward0>)
自动求导 y.grad: tensor([0.6333, 1.2666, 1.9000, 2.5333])
手算期望 y_grad: tensor([0.6333, 1.2666, 1.9000, 2.5333], grad_fn=<MulBackward0>)


In [27]:
node_q = z.grad_fn.next_functions[0][0]
node_x = node_q.next_functions[0][0]
node_y = node_q.next_functions[1][0]
print('z >', z.grad_fn.name())
print('q >', node_q.name())
print('x >', node_x.name())
print('y >', node_y.name())

z > SinBackward0
q > DotBackward0
x > struct torch::autograd::AccumulateGrad
y > struct torch::autograd::AccumulateGrad


# 第二阶段

In [32]:
x=torch.tensor(2.0, requires_grad=True)
y=torch.tensor(4.0, requires_grad=True)
z=torch.sin(x*y)
dzdx,dzdy=torch.autograd.grad(z, (x,y),create_graph=True)
print('dz/dx',dzdx)
print('dz/dy',dzdy)
d2zdx2=torch.autograd.grad(dzdx, (x,y), create_graph=True)
d2zdy2=torch.autograd.grad(dzdy, (x,y), create_graph=True)
print('d2z/dx/dy',d2zdx2)
print('d2z/dy/dx/dy',d2zdy2)

dz/dx tensor(-0.5820, grad_fn=<MulBackward0>)
dz/dy tensor(-0.2910, grad_fn=<MulBackward0>)
d2z/dx/dy (tensor(-15.8297, grad_fn=<MulBackward0>), tensor(-8.0604, grad_fn=<AddBackward0>))
d2z/dy/dx/dy (tensor(-8.0604, grad_fn=<AddBackward0>), tensor(-3.9574, grad_fn=<MulBackward0>))


# VJP

In [33]:
def vjp_func(x: Tensor, y: Tensor) -> Tensor:
    return x.dot(y).sin()
x=torch.arange(1.0, 5.0, requires_grad=True)
y=torch.arange(5.0, 9.0, requires_grad=True)
output=AF.vjp(vjp_func,(x,y))
print('func(x,y):', output[0])
print('VJP output:', output[1])

func(x,y): tensor(0.7739)
VJP output: (tensor([3.1666, 3.7999, 4.4332, 5.0666]), tensor([0.6333, 1.2666, 1.9000, 2.5333]))


In [38]:
z = x.dot(y).sin()
z.backward(retain_graph=True)

In [39]:
z.backward()

In [40]:
x.grad = None # Clear the gradient before starting
print('Before backward:', x.grad)
z1 = x.dot(y)
z1.backward()
print('After first backward:', x.grad)
z2 = x.dot(y)
z2.backward()
print('After second backward:', x.grad)

Before backward: None
After first backward: tensor([5., 6., 7., 8.])
After second backward: tensor([10., 12., 14., 16.])
